In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)


In [ ]:
data_pendapatan=pd.read_csv('../data/Pendapatan Mei - November 2023.csv',sep=',')
data_coa=pd.read_excel('../data/COA_GRUPING_REPORT (1).xlsx')
data_coa_459=pd.read_excel('../data/coa 4590901000.xlsx')
stacked_data_coa = pd.concat([data_coa, data_coa_459], ignore_index=True)
stacked_data_coa.to_excel('./../data/all_data_coa.xlsx')

In [ ]:
data_merge=data_pendapatan.merge(stacked_data_coa,left_on='GL Account',right_on='NOMOR_AKUN',how='left')
data_merge.to_csv("./../data/all_data_pendapatan_mei_november_2023.csv")

In [ ]:
data_merge

In [ ]:
unique_cust_no_per_name = data_merge.groupby('Cust Name')['Cust No.'].nunique()

multiple_cust_no = unique_cust_no_per_name[unique_cust_no_per_name > 1]

if multiple_cust_no.empty:
    print("Semua 'Cust Name' hanya memiliki satu 'Cust No.'")
else:
    print(f"Terdapat {len(multiple_cust_no)} 'Cust Name' yang memiliki lebih dari satu 'Cust No.'")
    print(multiple_cust_no)


In [ ]:
unique_cust_no_per_name = data_merge.groupby('Cust No.')['Cust Name'].nunique()

multiple_cust_no = unique_cust_no_per_name[unique_cust_no_per_name > 1]

if multiple_cust_no.empty:
    print("Semua 'Cust No' hanya memiliki satu 'Cust Name.'")
else:
    print(f"Terdapat {len(multiple_cust_no)} 'Cust No' yang memiliki lebih dari satu 'Cust Name.'")
    print(multiple_cust_no)


In [ ]:
cust_no_list = multiple_cust_no.index.tolist()

In [ ]:
for cust_no in cust_no_list:
    first_name = data_pendapatan.loc[data_pendapatan['Cust No.'] == cust_no, 'Cust Name'].iloc[0]
    data_merge.loc[data_pendapatan['Cust No.'] == cust_no, 'Cust Name'] = first_name

In [ ]:
unique_cust_no_per_name = data_merge.groupby('Cust No.')['Cust Name'].nunique()

multiple_cust_no = unique_cust_no_per_name[unique_cust_no_per_name > 1]

if multiple_cust_no.empty:
    print("Semua 'Cust No' hanya memiliki satu 'Cust Name.'")
else:
    print(f"Terdapat {len(multiple_cust_no)} 'Cust No' yang memiliki lebih dari satu 'Cust Name.'")
    print(multiple_cust_no)


# Checkpoint dump

In [ ]:
data_merge.to_csv("./../data/all_data_pendapatan_mei_november_2023.csv")

# End Joining

In [ ]:
unique_gl_no_per_name = data_merge.groupby(['Cust No.'])['GL Account'].nunique()

multiple_cust_no = unique_gl_no_per_name[unique_gl_no_per_name > 1]

if multiple_cust_no.empty:
    print("Semua kombinasi Cust No hanya memiliki satu GL Account")
else:
    print(f"Terdapat {len(multiple_cust_no)} kombinasi Cust No yang memiliki lebih dari satu GL Account")
    print(multiple_cust_no)


In [ ]:
data_pendapatan['Tanggal Nota'] = pd.to_datetime(data_pendapatan['Tanggal Nota'])
data_pendapatan['Tanggal Transaksi'] = data_pendapatan['Tanggal Nota'].dt.strftime('%m-%Y')

In [ ]:
data_pendapatan['Tanggal Transaksi'].unique()

In [ ]:
unique_combinations = data_pendapatan.groupby('Cust No.').ngroups
unique_combinations

In [ ]:
aggregate_cust_no = data_pendapatan.groupby(['Cust No.','Tanggal Transaksi']).agg(
    prof_center=('Prof Center', lambda x: list(x.unique())),
    count_prof_center=('Prof Center', 'nunique'),
    description=('Description',lambda x: list(x.unique())),
    cust_name=('Cust Name', lambda x: list(x.unique()) if len(x.unique()) > 1 else x.unique()[0]),
    nomor_nota=('Nomor Nota',lambda x: list(x.unique())),
    document_nomor_sap=('Document Number SAP',lambda x: list(x.unique())),
    tanggal=('Tanggal Nota', lambda x: [tuple(x.iloc[i:i+5]) for i in range(0, len(x), 1)]),
    nama_kapal=('Nama Kapal',lambda x:list(x.unique())),

    jumlah_transaksi_pembelian=('Pendapatan in IDR', lambda x: np.sum(x > 0)),
    jumlah_transaksi_retur=('Pendapatan in IDR', lambda x: np.sum(x < 0)),

    pembelian_median_pendapatan_idr=('Pendapatan in IDR', lambda x: np.median(x[x > 0]) if np.sum(x > 0) > 0 else 0),
    pembelian_min_pendapatan_idr=('Pendapatan in IDR', lambda x: np.min(x[x > 0]) if np.sum(x > 0) > 0 else 0),
    pembelian_max_pendapatan_idr=('Pendapatan in IDR', lambda x: np.max(x[x > 0]) if np.sum(x > 0) > 0 else 0),
    pembelian_sum_pendapatan_idr=('Pendapatan in IDR', lambda x: np.sum(x[x > 0])),

    retur_median_pendapatan_idr=('Pendapatan in IDR', lambda x: np.median(x[x < 0]) if np.sum(x < 0) > 0 else 0),
    retur_min_pendapatan_idr=('Pendapatan in IDR', lambda x: np.min(x[x < 0]) if np.sum(x < 0) > 0 else 0),
    retur_max_pendapatan_idr=('Pendapatan in IDR', lambda x: np.max(x[x < 0]) if np.sum(x < 0) > 0 else 0),
    retur_sum_pendapatan_idr=('Pendapatan in IDR', lambda x: np.sum(x[x < 0])),

    median_pendapatan_idr=('Pendapatan in IDR', 'median'),
    min_pendapatan_idr=('Pendapatan in IDR', 'min'),
    max_pendapatan_idr=('Pendapatan in IDR', 'max'),
    rata_rata_pendapatan_idr=('Pendapatan in IDR', 'mean'),
    sum_pendapatan_idr=('Pendapatan in IDR', 'sum'),

    median_pendapatan=('Pendapatan in USD', 'median'),
    min_pendapatan=('Pendapatan in USD', 'min'),
    max_pendapatan=('Pendapatan in USD', 'max'),
    rata_rata_pendapatan=('Pendapatan in USD', 'mean'),
    sum_pendapatan=('Pendapatan in USD', 'sum'),


    kontribusi_median_pendapatan=('Kontribusi in USD', 'median'),
    kontribusi_min_pendapatan=('Kontribusi in USD', 'min'),
    kontribusi_max_pendapatan=('Kontribusi in USD', 'max'),
    kontribusi_rata_rata_pendapatan=('Kontribusi in USD', 'mean'),
    sum_kontribusi=('Kontribusi in USD', 'sum'),

    kontribusi_median_pendapatan_idr=('Kontribusi in IDR', 'median'),
    kontribusi_min_pendapatan_idr=('Kontribusi in IDR', 'min'),
    kontribusi_max_pendapatan_idr=('Kontribusi in IDR', 'max'),
    kontribusi_rata_rata_pendapatan_idr=('Kontribusi in IDR', 'mean'),
    sum_kontribusi_idr=('Kontribusi in IDR', 'sum'),

    GTLOA=('GTLOA',lambda x:list(x.unique())),
    GT=('GT',lambda x:list(x.unique())),
    LOA=('LOA',lambda x:list(x.unique())),
    AP=('AP',lambda x:list(x.unique())),
    no_faktur=('No. Faktur',lambda x:list(x.unique())),
    nama_program=(' Nama Program',lambda x: list(x.unique()) if len(x.unique()) > 1 else x.unique()[0]),

)

In [ ]:
aggregate_cust_no

In [ ]:
aggregate_cust_no.reset_index(inplace=True)

In [ ]:
aggregate_cust_no

In [ ]:
aggregate_cust_no.to_excel("./../data/pendapatan_preprocess_v1-monthly-customer.xlsx")

In [ ]:
unique_cust_no_per_name = data_pendapatan.groupby('Cust No.')['GL Account'].nunique()

multiple_cust_no = unique_cust_no_per_name[unique_cust_no_per_name > 1]

if multiple_cust_no.empty:
    print("Semua 'Cust Name' hanya memiliki satu GL Account")
else:
    print(f"Terdapat {len(multiple_cust_no)} 'Cust Name' yang memiliki lebih dari satu GL Account")
    print(multiple_cust_no)


In [ ]:
aggregate_cust_no

In [ ]:
unique_cust_no_per_name = data_pendapatan.groupby('Cust No.')['Cust Name'].unique()
unique_cust_no_per_name_df = unique_cust_no_per_name.reset_index()

unique_cust_no_per_name_df['Cust Name'] = unique_cust_no_per_name_df['Cust Name'].apply(
    lambda x: x[0] if len(x) == 1 else x
)

In [ ]:
unique_cust_no_per_name_df.to_excel("./../data/mapping_customer_name_customer_no_pendapatan.xlsx")